In [1]:
"""
cd ~/llama.cpp && ./build/bin/llama-server \
  -m /home/user/.cache/huggingface/hub/models--unsloth--Qwen3.5-35B-A3B-GGUF/snapshots/bc014a17be43adabd7066b7a86075ff935c6a4e2/Qwen3.5-35B-A3B-Q4_K_M.gguf \
  --mmproj /home/user/.cache/huggingface/hub/models--unsloth--Qwen3.5-35B-A3B-GGUF/snapshots/bc014a17be43adabd7066b7a86075ff935c6a4e2/mmproj-F32.gguf \
  -ngl 999 \
  -c 81920 \
  --port 58080
"""

'\ncd ~/llama.cpp && ./build/bin/llama-server   -m /home/user/.cache/huggingface/hub/models--unsloth--Qwen3.5-35B-A3B-GGUF/snapshots/bc014a17be43adabd7066b7a86075ff935c6a4e2/Qwen3.5-35B-A3B-Q4_K_M.gguf   --mmproj /home/user/.cache/huggingface/hub/models--unsloth--Qwen3.5-35B-A3B-GGUF/snapshots/bc014a17be43adabd7066b7a86075ff935c6a4e2/mmproj-F32.gguf   -ngl 999   -c 81920   --port 58080\n'

In [2]:
from openai import OpenAI

# 로컬에 띄운 llama-server의 주소로 연결
client = OpenAI(
    base_url="http://127.0.0.1:58080/v1",
    api_key="llama.cpp" # 로컬 서버이므로 아무 문자열이나 넣어도 됩니다.
)

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image_url",
                "image_url": {
                    "url": "https://cdn.britannica.com/61/93061-050-99147DCE/Statue-of-Liberty-Island-New-York-Bay.jpg"
                }
            },
            {
                "type": "text",
                "text": "이 이미지를 묘사해줘."
            }
        ]
    }
]

response = client.chat.completions.create(
    model="Qwen/Qwen3.5-35B-A3B", # 이미 서버에 모델이 로드되어 있으므로 이름은 임의로 지정해도 됩니다.
    messages=messages,
    max_tokens=32768,
    temperature=0.1,
	top_p=0.8,
    presence_penalty=1.5,
    extra_body={
        "top_k": 20,
        "chat_template_kwargs": {"enable_thinking": False},     # 생각 스킵
    },
	# stream=True
)


# print(response)
print(response.choices[0].message.content)

이 이미지는 **뉴욕의 상징적인 풍경**을 담고 있습니다. 전경에는 **자유여신상 (Statue of Liberty)** 이 자유의 섬 (Liberty Island) 위에 우뚝 서 있으며, 녹색 청동으로 된 그녀의 모습은 햇살에 반짝이고 있습니다. 여신상은 오른손에 횃불을 들고 있고, 왼손에는 “1776”이 새겨진 헌법 문서를 들고 있는 것으로 보입니다.

배경으로는 **맨해튼의 스카이라인**이 펼쳐져 있으며, 다양한 고층 빌딩들이 하늘을 향해 솟아 있습니다. 특히 중앙에 보이는 **엠파이어 스테이트 빌딩**은 첨탑이 뚜렷하게 눈에 띕니다. 오른쪽에는 현대적인 유리 외관의 건물들 (예: 원 월드 트레이드 센터 근처의 빌딩들) 도 포함되어 있어 도시의 발전과 역사를 동시에 느낄 수 있습니다.

전체적으로 **맑고 푸른 하늘** 아래, 잔잔한 **허드슨 강 또는 뉴욕 항구** 물결이 자유여신상과 도시를 감싸고 있으며, 아침이나 오후의 따뜻한 햇빛이 전체 장면에 은은한 금빛을 더하고 있습니다. 왼쪽에는 작은 섬처럼 보이는 곳에 나무들이 있고, 미국 국기가 펄럭이고 있기도 합니다.

이 사진은 **미국 민주주의와 자유의 상징**인 자유여신상과 **세계 최대 도시 중 하나인 뉴욕의 번영**을 동시에 보여주는 매우 인상적인 풍경입니다.


In [3]:
from openai import OpenAI

# 로컬에 띄운 llama-server의 주소로 연결
client = OpenAI(
    base_url="http://127.0.0.1:58080/v1",
    api_key="llama.cpp" # 로컬 서버이므로 아무 문자열이나 넣어도 됩니다.
)

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image_url",
                "image_url": {
                    "url": "https://cdn.britannica.com/61/93061-050-99147DCE/Statue-of-Liberty-Island-New-York-Bay.jpg"
                }
            },
            {
                "type": "text",
                "text": "이 이미지를 묘사해줘."
            }
        ]
    }
]

response = client.chat.completions.create(
    model="Qwen/Qwen3.5-35B-A3B", # 이미 서버에 모델이 로드되어 있으므로 이름은 임의로 지정해도 됩니다.
    messages=messages,
    max_tokens=32768,
    temperature=0.1,
	top_p=0.8,
    presence_penalty=1.5,
    extra_body={
        "top_k": 20,
        "chat_template_kwargs": {"enable_thinking": True},
        # "chat_template_kwargs": {"enable_thinking": False},
    },
	stream=True
)

is_thinking = False

# 2. 스트리밍으로 넘어오는 조각(chunk)들을 반복문으로 받아서 실시간 출력합니다.
for chunk in response:
    delta = chunk.choices[0].delta
    
    # [생각 과정] 출력 부분 (reasoning_content)
    if getattr(delta, "reasoning_content", None):
        if not is_thinking:
            print("\n=== 모델의 생각 과정 ===\n", end="")
            is_thinking = True
            
        # end=""와 flush=True를 써야 줄바꿈 없이 한 글자씩 타자 치듯 출력됩니다.
        print(delta.reasoning_content, end="", flush=True)

    # [최종 답변] 출력 부분 (content)
    if getattr(delta, "content", None):
        if is_thinking:
            print("\n\n=== 답변 ===\n", end="")
            is_thinking = False
            
        print(delta.content, end="", flush=True)

print() # 마지막에 줄바꿈 한 번 추가


=== 모델의 생각 과정 ===
사용자가 제공한 이미지에 대해 설명을 요청했습니다. 이 이미지는 매우 유명한 뉴욕의 전경입니다.

1.  **주요 피사체 식별:**
    *   전경: 자유의 여신상 (Statue of Liberty). 리버티 섬에 서 있습니다.
    *   배경: 뉴욕 맨해튼의 스카이라인 (Manhattan Skyline). 다양한 고층 빌딩들이 보입니다. 특히 엠파이어 스테이트 빌딩이 눈에 띕니다.
    *   중간: 수면 (뉴욕 항구/허드슨 강).

2.  **세부 사항 관찰:**
    *   **자유의 여신상:** 녹색 청동 색상을 하고 있으며, 오른손에 횃불을 들고 있습니다. 기단부는 석재로 되어 있고, 주변에는 나무와 잔디가 보입니다.
    *   **스카이라인:** 다양한 높이와 디자인의 빌딩들이 빽빽하게 모여 있습니다. 햇빛이 비추어 건물들이 금색/주황색 빛을 띠고 있어 아침이나 저녁 시간대 (골든 아워) 로 추정됩니다.
    *   **하늘:** 맑고 푸른 하늘입니다. 구름은 거의 없습니다.
    *   **물:** 잔잔한 파도가 일고 있으며, 물결에 반사되는 빛이 보입니다. 작은 배나 부표가 몇 개 보입니다.

3.  **구도 및 분위기:**
    *   자유의 여신상이 왼쪽 중앙에 위치하여 시선을 끕니다.
    *   배경의 도시 풍경이 깊이감을 줍니다.
    *   전체적으로 평화롭고 웅장한 분위기를 자아냅니다.

4.  **묘사 작성 (한국어):**
    *   시작: 이 이미지는 뉴욕의 상징적인 풍경을 보여줍니다.
    *   전경 설명: 왼쪽에는 자유의 여신상이 리버티 섬 위에 우뚝 서 있습니다. 그녀의 녹색 외투와 횃불이 선명합니다. 기단부와 주변 공원도 보입니다.
    *   배경 설명: 뒤쪽으로는 뉴욕 맨해튼의 화려한 스카이라인이 펼쳐져 있습니다. 엠파이어 스테이트 빌딩을 포함한 수많은 마천루들이 햇살을 받아 빛나고 있습니다.
    *   환경 설명: 앞에는 넓은 수면 (뉴욕 항구) 이

In [4]:
prompt = "Pinky Pro 로봇에 장착된 카메라의 해상도는 얼마인가요?"

